In [3]:
import re
import numpy as np
import pandas as pd
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty
import joblib

# 1. Load your model components
print("Loading model components...")
model = joblib.load('model/xgb_k_model.pkl')
scaler = joblib.load('model/scaler.pkl')
feat_order = joblib.load('model/feature_order.pkl')
featurizer = ElementProperty.from_preset(preset_name="magpie")
print("✅ Successfully loaded!")

# 2. Create feature short name function and intervals
def short_name(s):
    s = re.sub(r'^MagpieData\s+', '', s)
    parts = s.split()
    if len(parts) >= 2:
        stat_type, prop_name = parts[0], ' '.join(parts[1:])
        stat_abbr = {'maximum':'max', 'minimum':'min', 'avg_dev':'avg_dev', 'mean':'mean'}.get(stat_type, stat_type)
        clean_prop = prop_name.replace(' ', '').replace('_', '')
        prop_abbr = clean_prop[:2] + clean_prop[-1] if len(clean_prop) >= 3 else clean_prop
        return f"{stat_abbr}_{prop_abbr}"
    return (s[:10] + '…') if len(s) > 10 else s

intervals = {
    'max_GSa': [-0.888, 4.628], 'max_MeT': [-2.011, 0.188], 'max_NVe': [0.035, 1.908],
    'mean_Row': [-0.855, 2.260], 'avg_dev_Nfd': [-0.228, 6.408], 'avg_dev_Mer': [-0.088, 1.712],
    'max_GSm': [-0.513, -0.513], 'min_GSa': [-0.422, 3.008],
}

# 3. Prediction function
def predict_single_material(formula, temp):
    """Predict the thermal conductivity of a single material"""
    print(f"\nAnalyzing material: {formula} at {temp} K...")

    try:
        # Construct input Dataframe
        input_df = pd.DataFrame({'formula': [formula], 'T(K)': [temp]})
        input_df['composition'] = input_df['formula'].apply(lambda x: Composition(x))

        # Feature extraction
        df_features = featurizer.featurize_dataframe(input_df, col_id='composition', ignore_errors=True)

        # Align and standardize
        X_new = df_features[feat_order].fillna(df_features[feat_order].mean())
        X_new_scaled = scaler.transform(X_new)

        # Predict
        kl_pred = model.predict(X_new_scaled)[0]

        # Interval matching statistics
        long_names = scaler.get_feature_names_out()
        short_names = [short_name(n) for n in long_names]
        intv_order = [fn for fn in intervals.keys() if fn in short_names]
        lows, highs = np.array([intervals[k][0] for k in intv_order]), np.array([intervals[k][1] for k in intv_order])
        idx_map = {fn: short_names.index(fn) for fn in intv_order}
        X10 = X_new_scaled[:, [idx_map[fn] for fn in intv_order]]
        hit_mat = (X10 >= lows) & (X10 <= highs)
        hits = hit_mat.sum(axis=1)[0]
        hit_feats = [intv_order[idx] for idx in np.where(hit_mat[0])[0]]

        # Print output results beautifully
        print("\n" + "="*50)
        print(f"Prediction successful! Results:")
        print(f"Material Formula:   {formula}")
        print(f"Testing Temp:       {temp} K")
        print(f"Predicted k:        {kl_pred:.4f} W/mK")
        print(f"Interval Hits:      {hits} / {len(intv_order)}")
        print(f"Specific Hit Feats: {', '.join(hit_feats) if hit_feats else 'None'}")
        print("="*50 + "\n")

        return kl_pred

    except Exception as e:
        print(f"❌ Prediction failed. Please check if the formula format is correct. Error: {e}\n")
        return None

"""# 4. Main input loop
print("\n" + "="*60)
print("     Material Thermal Conductivity Predictor")
print("="*60)
print("\nInstructions:")
print("  • Enter material formula (e.g., Ti0.99Nb0.01NiSn)")
print("  • Enter temperature in Kelvin (e.g., 400)")
print("  • Enter 'q' or 'exit' to finish input and predict")
print("-"*60)

# Initialize lists to store inputs
formulas = []
temperatures = []
count = 1

# Input loop
while True:
    print(f"\n📝 [Data Set {count}]")

    # Get formula input
    formula_input = input(f"   Formula: ").strip()

    # Check if user wants to exit
    if formula_input.lower() in ['q', 'exit']:
        if len(formulas) == 0:
            print("\n❌ No valid data entered, program exited safely.")
            exit()
        else:
            print(f"\n✅ Entered {len(formulas)} sets of data, starting prediction...")
            break

    # Validate formula is not empty
    if not formula_input:
        print("   ⚠️ Formula cannot be empty! Please re-enter this set.")
        continue

    # Get temperature input with validation
    try:
        temp_input = input(f"   Temperature T(K): ").strip()
        t_value = float(temp_input)

        # Check temperature is positive
        if t_value <= 0:
            print("   ⚠️ Temperature must be a positive number! Please re-enter this set.")
            continue

    except ValueError:
        print("   ⚠️ Invalid temperature format (please enter a number)! Please re-enter this set.")
        continue

    # Save data
    formulas.append(formula_input)
    temperatures.append(t_value)
    print(f"   ✅ Saved Data Set {count}: {formula_input} @ {t_value}K")
    count += 1
"""
formulas = ["ZnO","Zn0.96Al0.04O","Zn0.96Eu0.04O","Zn0.92Al0.04Eu0.04O"]
temperatures = [400,400,400,400]
count = len(formulas)
# 5. Perform predictions for all input materials
print("\n" + "="*60)
print("                Starting Predictions")
print("="*60)

results = []
for i, (formula, temp) in enumerate(zip(formulas, temperatures), 1):
    print(f"\n🔬 Processing [{i}/{len(formulas)}]")
    pred_k = predict_single_material(formula, temp)
    results.append(pred_k)

# 6. Summary report
print("\n" + "="*60)
print("                 Prediction Summary")
print("="*60)
print(f"\n{'No.':<5} {'Formula':<30} {'Temp (K)':<10} {'k (W/mK)':<12} {'Status':<10}")
print("-"*70)

for i, (formula, temp, pred_k) in enumerate(zip(formulas, temperatures, results), 1):
    status = "✅ Success" if pred_k is not None else "❌ Failed"
    pred_str = f"{pred_k:.4f}" if pred_k is not None else "N/A"
    print(f"{i:<5} {formula:<30} {temp:<10} {pred_str:<12} {status:<10}")

print("\n" + "="*60)
print("                  Prediction Completed!")
print("="*60)

Loading model components...
✅ Successfully loaded!

                Starting Predictions

🔬 Processing [1/4]

Analyzing material: ZnO at 400 K...


ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]


Prediction successful! Results:
Material Formula:   ZnO
Testing Temp:       400 K
Predicted k:        7.0548 W/mK
Interval Hits:      2 / 8
Specific Hit Feats: max_MeT, avg_dev_Nfd


🔬 Processing [2/4]

Analyzing material: Zn0.96Al0.04O at 400 K...


ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]


Prediction successful! Results:
Material Formula:   Zn0.96Al0.04O
Testing Temp:       400 K
Predicted k:        5.1900 W/mK
Interval Hits:      2 / 8
Specific Hit Feats: max_MeT, avg_dev_Nfd


🔬 Processing [3/4]

Analyzing material: Zn0.96Eu0.04O at 400 K...


ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]


Prediction successful! Results:
Material Formula:   Zn0.96Eu0.04O
Testing Temp:       400 K
Predicted k:        2.7838 W/mK
Interval Hits:      3 / 8
Specific Hit Feats: max_GSa, max_MeT, avg_dev_Nfd


🔬 Processing [4/4]

Analyzing material: Zn0.92Al0.04Eu0.04O at 400 K...


ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]


Prediction successful! Results:
Material Formula:   Zn0.92Al0.04Eu0.04O
Testing Temp:       400 K
Predicted k:        1.9506 W/mK
Interval Hits:      3 / 8
Specific Hit Feats: max_GSa, max_MeT, avg_dev_Nfd


                 Prediction Summary

No.   Formula                        Temp (K)   k (W/mK)     Status    
----------------------------------------------------------------------
1     ZnO                            400        7.0548       ✅ Success 
2     Zn0.96Al0.04O                  400        5.1900       ✅ Success 
3     Zn0.96Eu0.04O                  400        2.7838       ✅ Success 
4     Zn0.92Al0.04Eu0.04O            400        1.9506       ✅ Success 

                  Prediction Completed!
